In [1]:
import pandas as pd
import torch

In [2]:
print("GPU available: ", torch.cuda.is_available())

GPU available:  False


In [3]:
df = pd.read_csv("clean_post.csv")

In [4]:
from transformers import pipeline
#!pip install transformers

In [5]:
sentiment_model = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length = 512,
    device = 0);

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

C:\Users\Nishk\anaconda3\envs\torch_gpu\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nishk\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [7]:
def get_sentiment(text):
    try:
        result = sentiment_model(str(text)[:512])[0]
        return result["label"], round(result["score"], 4)
    except:
        return "neutral", 0.0

In [8]:
print("Running sentiment analysis... this will take a few minutes")
df[["sentiment", "confidence"]] = df["clean_text"].apply(
    lambda x: pd.Series(get_sentiment(x))
)

Running sentiment analysis... this will take a few minutes


In [10]:
print("\nSentiment Distribution: ")
print(df["sentiment"].value_counts())
print("\nAverage confidence per sentiment: ")
print(df.groupby("sentiment")["confidence"].mean().round(3))
df.to_csv("sentiment_posts.csv")


Sentiment Distribution: 
sentiment
negative    1311
neutral      661
positive     119
Name: count, dtype: int64

Average confidence per sentiment: 
sentiment
negative    0.748
neutral     0.682
positive    0.705
Name: confidence, dtype: float64


In [11]:
print(df.head())

   Unnamed: 0       id                                              title  \
0           0  1rlblhn  President Bought Netflix Debt in January 2026,...   
1           1  1rl6lay  US air defenses may not be able to intercept m...   
2           2  1rlhffx  GOP state lawmakers urge White House to halt e...   
3           3  1rkv7az  Brendan Carr Can’t Explain Why ‘Equal Time’ Ru...   
4           4  1rlejk6  I was at a QuitGPT protest, and the discontent...   

  body  score  upvote_ratio  num_comments                    flair  \
0  NaN   8871          0.97           568                 Business   
1  NaN   5598          0.95           814                 Business   
2  NaN    732          0.97            52  Artificial Intelligence   
3  NaN  23944          0.96           401                 Politics   
4  NaN    790          0.96            22  Artificial Intelligence   

                 author                                                url  \
0            ControlCAD  https://www.h